## 导入实验所需库

本部分导入实现神经机器翻译所需的Python库，包括：

- torch：深度学习框架
- torch.nn：神经网络模块
- torch.optim：优化器
- torchtext：文本处理工具
- spacy：自然语言处理工具
- numpy：数值计算工具
- random：随机数生成工具

这些库将在后续模型构建和训练过程中使用。

In [124]:
import os
import datasets
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset

from torchtext.datasets import Multi30k
from torchtext.data import Field, BucketIterator

import torchtext
from torchtext.vocab import build_vocab_from_iterator 

import spacy
import numpy as np

import random
import math
import time

## 设置随机种子

为了保证实验结果具有可重复性，需要固定随机种子。

随机种子将作用于：

- Python随机模块
- NumPy随机模块
- PyTorch随机模块

这样每次运行模型时的数据划分、参数初始化及训练结果基本保持一致。

In [126]:
SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [128]:
spacy_de = spacy.load('de_core_news_sm')
spacy_en = spacy.load('en_core_web_sm')

# 数据集 Dataset

## 1、添加一个Markdown单元格，在其中解释下方单元格的两行代码

设置 `os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'` ，这样做具体改变了什么？ 为什么要设置`HF_ENDPOINT='https://hf-mirror.com'`而非直接使用官方源？

`dataset = datasets.load_dataset("bentrevett/multi30k")` 这行代码具体完成了什么操作？

In [ ]:
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
dataset = datasets.load_dataset("bentrevett/multi30k")

1. `os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'`
该代码用于修改系统环境变量，将 Hugging Face 资源的默认访问地址切换为国内镜像站。
由于 Hugging Face 官方服务器在境外，国内直接访问存在网速慢、连接超时、下载失败等问题，使用国内镜像可以大幅提升数据集、模型的下载速度与稳定性，保证代码正常运行。

2. `dataset = datasets.load_dataset("bentrevett/multi30k")`
该行代码从 Hugging Face 数据集仓库中加载 **multi30k 德英平行机器翻译数据集**，自动完成数据集下载、缓存与解析，最终返回一个包含训练集、验证集、测试集的数据集对象，后续可直接用于模型训练与测试。

## 2、运行下方的单元格。

你会看到数据集对象（一个DatasetDict）包含训练、验证和测试集，每个集合中的样本数量，以及每个集合中的特征（“en”和“de”）。

In [145]:
dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [147]:
train_data, valid_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"],
)

## 3、运行下方的单元格。

我们可以索引每个数据集来查看单个示例。每个例子都有两个特征：“en”和“de”，是对应的英语和德语。

In [153]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

接下来我们进行分词。英语/德语的分词较中文要直接，比如句子"good morning!会被分词为["good", "morning", "!"]序列。

下方的代码要成功安装en_core_web_sm和de_core_news_sm后才不会报错。

# 加载分词器 Tokenizers

In [ ]:
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

## 4、运行下方的单元格。

我们可以使用.tokenizer方法调用每个spaCy模型的分词器，该方法接受字符串并返回Token对象序列。我们可以使用text属性从Token对象中获取字符串。

In [166]:
string = "What a lovely day it is today!"
[token.text for token in en_nlp.tokenizer(string)]

['What', 'a', 'lovely', 'day', 'it', 'is', 'today', '!']

## 5、添加一个Markdown单元格，在其中解释下方单元格的函数的作用。

In [169]:
def tokenize_example(example, en_nlp, de_nlp, max_length, lower, sos_token, eos_token):
    en_tokens = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
    de_tokens = [token.text for token in de_nlp.tokenizer(example["de"])][:max_length]
    if lower:
        en_tokens = [token.lower() for token in en_tokens]
        de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

## 6、添加一个Markdown单元格，在其中解释下方单元格出现的<sos>和<eos>的含义，以及map函数的作用。

In [177]:
max_length = 1000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

train_data = train_data.map(tokenize_example, fn_kwargs=fn_kwargs)
valid_data = valid_data.map(tokenize_example, fn_kwargs=fn_kwargs)
test_data = test_data.map(tokenize_example, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

tokenize_example 函数说明
tokenize_example 是德英翻译任务的数据预处理函数，用于对平行语料进行分词和标准化处理。

## 7、运行下方的单元格

重新打印train_data[0]，验证小写字符串列表以及序列标记的开始/结束符已被成功添加。

In [182]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# 构建词表 Vocabularies

下一个步骤是为源语言和目标语言构建词汇表，将词语映射为数字索引。

In [204]:
from collections import Counter

min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"
sos_token = "<sos>"
eos_token = "<eos>"

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

en_counter = Counter()
for tokens in train_data["en_tokens"]:
    en_counter.update(tokens)

filtered_en_tokens = [token for token, count in en_counter.items() if count >= min_freq]

final_en_tokens = special_tokens + filtered_en_tokens

en_vocab = torchtext.vocab.build_vocab_from_iterator(
    iter([final_en_tokens])
)

de_counter = Counter()
for tokens in train_data["de_tokens"]:
    de_counter.update(tokens)

filtered_de_tokens = [token for token, count in de_counter.items() if count >= min_freq]
final_de_tokens = special_tokens + filtered_de_tokens

de_vocab = torchtext.vocab.build_vocab_from_iterator(
    iter([final_de_tokens])
)

unk_index = en_vocab.stoi[unk_token]
pad_index = en_vocab.stoi[pad_token]

def en_lookup_token(token):
    return en_vocab.stoi.get(token, unk_index)
en_vocab.lookup_token = en_lookup_token

def de_lookup_token(token):
    return de_vocab.stoi.get(token, unk_index)
de_vocab.lookup_token = de_lookup_token

1lines [00:00, 971.13lines/s]
1lines [00:00, 473.83lines/s]


## 8、运行下方两个单元格

验证词汇表，分别打印英语词汇表和德语词汇表的前十个Token。

In [212]:
list(en_vocab.stoi.keys())[:10]

['<unk>', '<pad>', '<eos>', '<sos>', ' ', '!', '"', '#', '%', '&']

In [214]:
list(de_vocab.stoi.keys())[:10]

['<unk>', '<pad>', '<eos>', '<sos>', ' ', '!', '"', '#', '%', '&']

## 9、运行下方的单元格

使用get_stoi（stoi = "string to int "）方法获取指定的Token的索引。

In [218]:
en_vocab["the"]

5247

设置默认未知词索引

In [228]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]

def en_lookup_token(token):
    return en_vocab.stoi.get(token, unk_index)
en_vocab.lookup_token = en_lookup_token

def de_lookup_token(token):
    return de_vocab.stoi.get(token, unk_index)
de_vocab.lookup_token = de_lookup_token

词汇表的另一个有用特性是lookup_indices方法。它接受一个Token列表并返回一个索引列表。

## 10、运行下方的单元格

将token列表转为索引列表

In [235]:
tokens = ["i", "love", "watching", "crime", "shows"]
[en_vocab.stoi.get(token, unk_index) for token in tokens]

[2517, 2969, 5698, 0, 4542]

对应的，lookup_tokens方法使用词汇表将索引列表转换回Token列表。

## 11、运行下方的单元格

观察将索引列表转为token列表

In [257]:
indices = [en_vocab.stoi.get(token, unk_index) for token in tokens]
[en_vocab.itos[i] for i in indices]

['i', 'love', 'watching', '<unk>', 'shows']

## 12、添加一个Markdown单元格，在其中解释为什么原本的"crime"被转换成了<unk>。

在构建词汇表时，我们设置了 min_freq=2，即只保留在训练集中出现次数至少为 2 的词。如果某个词（如“crime”）在训练数据中出现的次数少于 2 次，它就不会被加入到词汇表中。当我们对验证集或测试集中的文本进行数值化（numericalize）时，遇到这种“未登录词”（out-of-vocabulary, OOV），就会用 标记（unknown token）来代替。因此，原本的“crime”被转换成了 ，表示模型不认识这个词。

## 13、添加一个Markdown单元格，在其中解释下方两个单元格中代码的作用。

In [262]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = en_vocab.lookup_indices(example["en_tokens"])
    de_ids = de_vocab.lookup_indices(example["de_tokens"])
    return {"en_ids": en_ids, "de_ids": de_ids}

In [266]:
def numericalize_example(example, en_vocab, de_vocab):
    en_ids = [en_vocab.stoi.get(token, unk_index) for token in example["en_tokens"]]
    de_ids = [de_vocab.stoi.get(token, unk_index) for token in example["de_tokens"]]
    return {"en_ids": en_ids, "de_ids": de_ids}

train_data = train_data.map(
    lambda x: numericalize_example(x, en_vocab=en_vocab, de_vocab=de_vocab)
)
valid_data = valid_data.map(
    lambda x: numericalize_example(x, en_vocab=en_vocab, de_vocab=de_vocab)
)
test_data = test_data.map(
    lambda x: numericalize_example(x, en_vocab=en_vocab, de_vocab=de_vocab)
)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

### 代码功能说明

这两段代码的作用是**将文本数据转换为模型可处理的数字索引格式**，也就是数据的「数值化」处理。

1.  **`numericalize_example` 函数**
    - 定义了如何将单条数据中的 token 列表转为数字索引列表。
    - 接收单条数据 `example`、英语词表 `en_vocab` 和德语词表 `de_vocab`。
    - 使用 `lookup_indices` 方法，分别将 `en_tokens` 和 `de_tokens` 转换为对应的索引列表 `en_ids` 和 `de_ids`。
    - 返回包含新字段 `en_ids` 和 `de_ids` 的字典，用于更新数据集。

2.  **批量应用数值化**
    - `fn_kwargs` 定义了要传递给 `map` 函数的额外参数，即英语和德语词表。
    - 通过 `train_data.map()`、`valid_data.map()`、`test_data.map()`，将 `numericalize_example` 函数应用到训练集、验证集和测试集的每一条数据上。
    - 最终所有数据都会新增 `en_ids` 和 `de_ids` 列，为后续构建 DataLoader 和模型训练做准备。

## 14、运行下方的单元格

重新打印train_data[0]，验证"en_ids" and "de_ids"被成功添加。

In [271]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [3, 5497, 5878, 14, 5770, 3007, 214, 3462, 3299, 3024, 745, 16, 2],
 'de_ids': [3,
  7747,
  3436,
  7442,
  4513,
  5974,
  3298,
  2170,
  3304,
  1410,
  4668,
  7174,
  1253,
  15,
  2]}

Dataset类为我们处理的另一件事是将features转换为正确的类型。每个例子中的索引目前都是基本的Python整数。然而，为了在PyTorch中使用它们，它们需要转换为PyTorch张量。with_format方法将columns参数转换为给定的类型。这里，我们指定类型为“torch”，columns为“en_ids”和“de_ids”（我们想要转换为PyTorch张量的features）。默认情况下，with_format将删除任何不在传递给列的features列表中的features。我们希望保留这些features，这可以通过output_all_columns=True来实现。

In [277]:
data_type = "torch"
format_columns = ["en_ids", "de_ids"]

train_data = train_data.with_format(
    type=data_type, columns=format_columns, output_all_columns=True
)

valid_data = valid_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

test_data = test_data.with_format(
    type=data_type,
    columns=format_columns,
    output_all_columns=True,
)

## 15、运行下方的单元格

重新打印train_data[0]，验证“en_ids”和“de_ids”特征被转换为了张量。

In [280]:
train_data[0]

{'en_ids': tensor([   3, 5497, 5878,   14, 5770, 3007,  214, 3462, 3299, 3024,  745,   16,
            2]),
 'de_ids': tensor([   3, 7747, 3436, 7442, 4513, 5974, 3298, 2170, 3304, 1410, 4668, 7174,
         1253,   15,    2]),
 'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

# Data Loaders

数据准备的最后一步是创建Data Loaders。可以对它们进行迭代以返回一批数据，每一批数据都是一个字典，其中包含数字化的英语和德语句子作为PyTorch张量。

## 16、添加一个Markdown单元格，在其中解释下方两个单元格中的函数的作用。

In [288]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch
    return collate_fn

定义数据加载器

In [290]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

生成训练、验证、测试数据加载器

In [292]:
batch_size = 128

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

# 构建模型
我们将分三部分构建模型。编码器，解码器和封装编码器和解码器的seq2seq模型。

# 编码器 Encoder
首先是编码器，它是一个2层的LSTM。

## 17、添加一个Markdown单元格，解释下方单元格中Encoder类的代码。
包括输入参数，核心组件（词嵌入层、LSTM层、Dropout层），forwad函数的处理流程，和输出。

In [295]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src = [src length, batch size]
        embedded = self.dropout(self.embedding(src))
        # embedded = [src length, batch size, embedding dim]
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs = [src length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # outputs are always from the top hidden layer
        return hidden, cell

## Encoder 类说明

Encoder 是 Seq2Seq 模型的编码器，负责把德语句子编码成语义向量，送给解码器生成英语句子。

---

### 初始化方法 `__init__`

- `input_dim`：德语词汇表大小
- `embedding_dim`：词向量维度
- `hidden_dim`：LSTM 隐藏层维度
- `n_layers`：LSTM 层数
- `dropout`：防止过拟合的随机失活率

内部构建了三层结构：词嵌入层将索引转为向量，LSTM 层处理序列信息，Dropout 层随机失活神经元。

---

### 前向传播 `forward`

输入是德语句子的索引序列。先经过嵌入层和 Dropout 处理，再输入 LSTM 进行编码。最终返回 LSTM 最后的隐藏状态和细胞状态，这两个向量包含整句德语的语义信息，作为解码器的初始输入。

## 定义解码器Decoder

接下来是解码器，它需要与编码器对齐，同样是一个2层的LSTM。

## 18、添加一个Markdown单元格，描述Decoder的工作流程。

In [299]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # n directions in the decoder will both always be 1, therefore:
        # hidden = [n layers, batch size, hidden dim]
        # context = [n layers, batch size, hidden dim]
        input = input.unsqueeze(0)
        # input = [1, batch size]
        embedded = self.dropout(self.embedding(input))
        # embedded = [1, batch size, embedding dim]
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output = [seq length, batch size, hidden dim * n directions]
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # seq length and n directions will always be 1 in this decoder, therefore:
        # output = [1, batch size, hidden dim]
        # hidden = [n layers, batch size, hidden dim]
        # cell = [n layers, batch size, hidden dim]
        prediction = self.fc_out(output.squeeze(0))
        # prediction = [batch size, output dim]
        return prediction, hidden, cell

Decoder 是 Seq2Seq 模型的解码器，负责根据编码器的语义向量，逐词生成目标语言（英语）句子。

---

前向传播 `forward`
每个时间步接收一个目标词，结合编码器传来的初始隐藏状态，通过 LSTM 更新状态，并输出下一个词的概率分布。

训练阶段采用「教师强制」策略，即每一步都输入真实的目标词；而推理阶段则使用上一步的预测结果作为下一步的输入。

返回的 `prediction` 用于计算损失，更新后的 `(hidden, cell)` 会传递给下一个时间步，继续生成序列。

# Seq2Seq

## 19、添加一个Markdown单元格，解释下方单元格中Seq2Seq类的代码。
包括forward函数的流程，以及teacher forcing机制。

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        assert (
            encoder.hidden_dim == decoder.hidden_dim
        ), "Hidden dimensions of encoder and decoder must be equal!"
        assert (
            encoder.n_layers == decoder.n_layers
        ), "Encoder and decoder must have equal number of layers!"

    def forward(self, src, trg, teacher_forcing_ratio):
        # src = [src length, batch size]
        # trg = [trg length, batch size]
        # teacher_forcing_ratio is probability to use teacher forcing
        # e.g. if teacher_forcing_ratio is 0.75 we use ground-truth inputs 75% of the time
        batch_size = trg.shape[1]
        trg_length = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        # tensor to store decoder outputs
        outputs = torch.zeros(trg_length, batch_size, trg_vocab_size).to(self.device)
        # last hidden state of the encoder is used as the initial hidden state of the decoder
        hidden, cell = self.encoder(src)
        # hidden = [n layers * n directions, batch size, hidden dim]
        # cell = [n layers * n directions, batch size, hidden dim]
        # first input to the decoder is the <sos> tokens
        input = trg[0, :]
        # input = [batch size]
        for t in range(1, trg_length):
            # insert input token embedding, previous hidden and previous cell states
            # receive output tensor (predictions) and new hidden and cell states
            output, hidden, cell = self.decoder(input, hidden, cell)
            # output = [batch size, output dim]
            # hidden = [n layers, batch size, hidden dim]
            # cell = [n layers, batch size, hidden dim]
            # place predictions in a tensor holding predictions for each token
            outputs[t] = output
            # decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio
            # get the highest predicted token from our predictions
            top1 = output.argmax(1)
            # if teacher forcing, use actual next token as next input
            # if not, use predicted token
            input = trg[t] if teacher_force else top1
        return outputs

### Seq2Seq 模型 `forward` 流程

---

1.  **编码阶段**
    将源语言序列 `src` 输入编码器，得到最终的隐藏状态 `hidden` 和细胞状态 `cell`，这两个向量将作为解码器的初始状态。

2.  **初始化阶段**
    - 创建 `outputs` 张量，用于存储解码器在每个时间步的预测结果。
    - 解码器的第一个输入为目标序列的起始标记 `<sos>`（取自目标序列的第一个词）。

3.  **循环解码阶段**
    从第 2 个时间步（索引为 1）开始，循环执行到目标序列的最大长度，每个时间步执行以下操作：
    - 将当前输入 `input` 和状态 `(hidden, cell)` 送入解码器，得到当前时间步的预测 `output` 和更新后的状态。
    - 将 `output` 存入 `outputs[t]`。
    - **教师强制决策**：以概率 `teacher_forcing_ratio` 选择真实目标词 `trg[t]` 作为下一步的输入；否则，使用当前预测概率最高的词 `top1` 作为下一步输入。

4.  **返回结果**
    返回包含所有时间步预测结果的张量 `outputs`。（注：第一个时间步的输出通常不参与损失计算，因此未被使用。）

# 初始化模型

## 20、添加注释
分别将“# 编码器初始化”，“# 解码器初始化”，“# Seq2Seq模型整合”这三行注释加到下方单元格中正确的位置

In [314]:
import torch
import torch.nn as nn

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: [src_len, batch_size]
        # trg: [trg_len, batch_size]
        
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        hidden, cell = self.encoder(src)

        input = trg[0, :]

        for t in range(1, trg_len):
  
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output

            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1

        return outputs
        
model = Seq2Seq(encoder, decoder, device).to(device)

初始化模型权重

In [316]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(7853, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(5893, 256)
    (rnn): LSTM(256, 512, num_layers=2, dropout=0.5)
    (fc_out): Linear(in_features=512, out_features=5893, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [318]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 13,898,501 trainable parameters


定义优化器optimizer

In [321]:
optimizer = optim.Adam(model.parameters())

定义损失函数Loss Function

In [324]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_index)

Training Loop:

## 21、给下方单元格中的代码逐行加注释

In [328]:
def train_fn(
    model, data_loader, optimizer, criterion, clip, teacher_forcing_ratio, device
):
    # 将模型切换为训练模式（启用 Dropout、BatchNorm 等训练专属层）
    model.train()
    # 初始化本轮 epoch 的总损失为 0
    epoch_loss = 0
    # 遍历数据加载器中的每一个 batch
    for i, batch in enumerate(data_loader):
        # 将源语言（德语）的索引序列放到指定设备（GPU/CPU）
        src = batch["de_ids"].to(device)
        # 将目标语言（英语）的索引序列放到指定设备
        trg = batch["en_ids"].to(device)

        # 梯度清零，避免上一轮梯度累积影响本轮训练
        optimizer.zero_grad()
        # 前向传播：输入源序列、目标序列，启用 Teacher Forcing，得到模型预测输出
        output = model(src, trg, teacher_forcing_ratio)

        # 提取输出的最后一维（目标语言词汇表大小）
        output_dim = output.shape[-1]
        # 调整输出形状：去掉 <sos> 起始 token，展平为二维张量，适配损失函数输入
        output = output[1:].view(-1, output_dim)
        # 调整目标序列形状：去掉 <sos>，展平为一维张量，与输出对齐
        trg = trg[1:].view(-1)

        # 计算预测值与真实值的损失
        loss = criterion(output, trg)
        # 反向传播，计算梯度
        loss.backward()
        # 梯度裁剪，防止梯度爆炸，clip 为最大梯度范数
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        # 更新模型参数
        optimizer.step()

        # 累加本轮 batch 的损失到 epoch 总损失
        epoch_loss += loss.item()
    # 返回本轮 epoch 的平均损失（总损失 / batch 数量）
    return epoch_loss / len(data_loader)

Evaluation Loop:

In [331]:
def evaluate_fn(model, data_loader, criterion, device):
    # 将模型切换为评估模式（禁用 Dropout、BatchNorm 等，保证推理稳定）
    model.eval()
    # 初始化本轮 epoch 的总损失为 0
    epoch_loss = 0
    # 禁用梯度计算，减少内存占用、加速推理（评估无需更新参数）
    with torch.no_grad():
        # 遍历数据加载器中的每一个 batch
        for i, batch in enumerate(data_loader):
            # 将源语言（德语）的索引序列放到指定设备
            src = batch["de_ids"].to(device)
            # 将目标语言（英语）的索引序列放到指定设备
            trg = batch["en_ids"].to(device)

            # 前向传播：关闭 Teacher Forcing（teacher_forcing_ratio=0），完全自回归生成
            output = model(src, trg, 0)

            # 提取输出的最后一维（目标语言词汇表大小）
            output_dim = output.shape[-1]
            # 调整输出形状：去掉 <sos>，展平为二维张量
            output = output[1:].view(-1, output_dim)
            # 调整目标序列形状：去掉 <sos>，展平为一维张量
            trg = trg[1:].view(-1)

            # 计算预测值与真实值的损失
            loss = criterion(output, trg)
            # 累加本轮 batch 的损失到 epoch 总损失
            epoch_loss += loss.item()
    # 返回本轮 epoch 的平均损失
    return epoch_loss / len(data_loader)

## 开始训练模型

In [347]:
import torch
import numpy as np
import torch.nn as nn
import tqdm  # 按你指定的方式导入

# 训练超参数
n_epochs = 1
clip = 1.0
teacher_forcing_ratio = 0.5
best_valid_loss = float("inf")

# 训练循环（按你要的 tqdm.tqdm 写法）
for epoch in tqdm.tqdm(range(n_epochs)):
    # 训练阶段
    train_loss = train_fn(
        model,
        train_data_loader,
        optimizer,
        criterion,
        clip,
        teacher_forcing_ratio,
        device,
    )

    # 验证阶段
    valid_loss = evaluate_fn(
        model,
        valid_data_loader,
        criterion,
        device,
    )

    # 保存最优模型
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "tut1-model.pt")

    # 打印日志
    print(f"\tTrain Loss: {train_loss:7.3f} | Train PPL: {np.exp(train_loss):7.3f}")
    print(f"\tValid Loss: {valid_loss:7.3f} | Valid PPL: {np.exp(valid_loss):7.3f}")

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:30<00:00, 30.51s/it]

	Train Loss:   5.038 | Train PPL: 154.134
	Valid Loss:   4.971 | Valid PPL: 144.202


## 模型验证

In [349]:
model.load_state_dict(torch.load("tut1-model.pt"))

<All keys matched successfully>

# 定义翻译函数

In [351]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=25,
):
    model.eval()
    with torch.no_grad():
        if isinstance(sentence, str):
            tokens = [token.text for token in de_nlp.tokenizer(sentence)]
        else:
            tokens = [token for token in sentence]
        if lower:
            tokens = [token.lower() for token in tokens]
        tokens = [sos_token] + tokens + [eos_token]
        ids = de_vocab.lookup_indices(tokens)
        tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)
        hidden, cell = model.encoder(tensor)
        inputs = en_vocab.lookup_indices([sos_token])
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]]).to(device)
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            if predicted_token == en_vocab[eos_token]:
                break
        tokens = en_vocab.lookup_tokens(inputs)
    return tokens

In [353]:
sentence = test_data[0]["de"]
expected_translation = test_data[0]["en"]

sentence, expected_translation

('Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.',
 'A man in an orange hat starring at something.')

In [361]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=50
):
    model.eval()

    if isinstance(sentence, str):
        tokens = [token.text for token in de_nlp.tokenizer(sentence)]
    else:
        tokens = [token for token in sentence]

    if lower:
        tokens = [token.lower() for token in tokens]

    tokens = [sos_token] + tokens + [eos_token]

    unk_idx = de_vocab.stoi.get("<unk>", 0)
    ids = [de_vocab.stoi.get(token, unk_idx) for token in tokens]

    tensor = torch.LongTensor(ids).unsqueeze(-1).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(tensor)

    inputs = [en_vocab.stoi[sos_token]]
    for _ in range(max_output_length):
        input_tensor = torch.LongTensor([inputs[-1]]).unsqueeze(-1).to(device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(input_tensor, hidden, cell)
        predicted_token = output.argmax(1).item()
        inputs.append(predicted_token)
        if predicted_token == en_vocab.stoi[eos_token]:
            break

    translation_tokens = [en_vocab.itos[i] for i in inputs]
    return translation_tokens[1:-1]

## 22、运行下方单元格，得到测试集第0个索引的翻译

因为epoch只进行了一轮，不会有好的效果的翻译。 感兴趣的同学可自行增加训练轮数，观察loss和翻译质量的变化

In [389]:
def translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=50
):
    model.eval()
    if isinstance(sentence, str):
        tokens = [token.text for token in de_nlp.tokenizer(sentence)]
    else:
        tokens = [token for token in sentence]

    if lower:
        tokens = [token.lower() for token in tokens]

    tokens = [sos_token] + tokens + [eos_token]
    unk_idx = de_vocab.stoi.get("<unk>", 0)
    ids = [de_vocab.stoi.get(token, unk_idx) for token in tokens]

    src_tensor = torch.LongTensor(ids).unsqueeze(1).to(device)
    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    trg_idx = en_vocab.stoi[sos_token]
    result = [trg_idx]

    for _ in range(max_output_length):
        trg_tensor = torch.LongTensor([trg_idx]).to(device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
        
        trg_idx = output.argmax(dim=1).item()
        result.append(trg_idx)
        
        if trg_idx == en_vocab.stoi[eos_token]:
            break

    tokens_out = [en_vocab.itos[i] for i in result]
    return tokens_out

sentence = "Ein Mann sitzt auf einer Bank."
translation = translate_sentence(
    sentence,
    model,
    en_nlp,
    de_nlp,
    en_vocab,
    de_vocab,
    lower,
    sos_token,
    eos_token,
    device,
    max_output_length=80
)

print("原始token序列:", translation)
print("翻译结果:", " ".join([t for t in translation if t not in [sos_token, eos_token]]))

原始token序列: ['<sos>', 'a', 'man', 'in', 'a', 'a', '.', '<eos>']
翻译结果: a man in a a .
